# Day 9 – Processed E-commerce Dataset



 Import Pandas and Load the Datasets

In [2]:
import pandas as pd

customers = pd.read_csv("Day9_Customers.csv")
products = pd.read_csv("Day9_Products.csv")
orders = pd.read_csv("Day9_Orders.csv")

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)


Customers: (30, 5)
Products: (20, 5)
Orders: (120, 7)


In [3]:
customers.head()


,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New


In [4]:
products.head()


,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro
2,P003,Wireless Mouse,Electronics,899,TechGear
3,P004,Smart Watch,Electronics,3299,FitTech
4,P005,Power Bank,Electronics,1199,VoltPlus


In [5]:
orders.head()


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


In [7]:
orders_part1 = orders.iloc[:len(orders)//2].copy()
orders_part2 = orders.iloc[len(orders)//2:].copy()

orders_combined = pd.concat([orders_part1, orders_part2], ignore_index=True)

print("Part 1:", orders_part1.shape)
print("Part 2:", orders_part2.shape)
print("After concat:", orders_combined.shape)


Part 1: (60, 7)
Part 2: (60, 7)
After concat: (120, 7)


 DateTime Operations


In [8]:
orders_combined["Order_Date"] = pd.to_datetime(orders_combined["Order_Date"])

orders_combined["Order_Year"] = orders_combined["Order_Date"].dt.year
orders_combined["Order_Month"] = orders_combined["Order_Date"].dt.month
orders_combined["Order_Month_Name"] = orders_combined["Order_Date"].dt.month_name()
orders_combined["Order_Day"] = orders_combined["Order_Date"].dt.day
orders_combined["Order_Day_Name"] = orders_combined["Order_Date"].dt.day_name()

orders_combined[[
    "Order_Date", "Order_Year", "Order_Month",
    "Order_Month_Name", "Order_Day", "Order_Day_Name"
]].head()


,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name
0,2026-02-19,2026,2,February,19,Thursday
1,2026-01-25,2026,1,January,25,Sunday
2,2026-02-26,2026,2,February,26,Thursday
3,2026-03-04,2026,3,March,4,Wednesday
4,2026-03-29,2026,3,March,29,Sunday


 Merge Customer and Product Information



In [9]:
processed = orders_combined.merge(
    customers,
    on="Customer_ID",
    how="left"
)

processed = processed.merge(
    products,
    on="Product_ID",
    how="left"
)

print("Processed shape after merges:", processed.shape)
processed.head()


Processed shape after merges: (120, 20)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name,Customer_Name,City,Region,Membership_Type,Product_Name,Category,Unit_Price,Brand
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,2026,2,February,19,Thursday,Harsh Vardhan,Noida,North,Premium,Cricket Bat,Sports,2499,BatPro
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,2026,1,January,25,Sunday,Ishita Gupta,Bengaluru,South,Premium,Wireless Mouse,Electronics,899,TechGear
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,2026,2,February,26,Thursday,Karan Joshi,Chandigarh,North,Regular,Smart Watch,Electronics,3299,FitTech
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,2026,3,March,4,Wednesday,Maryam Khan,Hyderabad,South,Regular,Machine Learning Basics,Books,999,AIPress
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,2026,3,March,29,Sunday,Reyansh Jain,Kolkata,East,New,Coffee Maker,Home & Kitchen,3499,HomeBrew


In [10]:
processed["Total_Amount"] = processed.apply(
    lambda row: row["Quantity"] * row["Unit_Price"],
    axis=1
)

processed["Order_Value_Category"] = processed["Total_Amount"].apply(
    lambda x: "High" if x >= 5000 else ("Medium" if x >= 2000 else "Low")
)

processed[[
    "Order_ID", "Product_Name", "Quantity",
    "Unit_Price", "Total_Amount", "Order_Value_Category"
]].head()


,Order_ID,Product_Name,Quantity,Unit_Price,Total_Amount,Order_Value_Category
0,O0001,Cricket Bat,2,2499,4998,Medium
1,O0002,Wireless Mouse,2,899,1798,Low
2,O0003,Smart Watch,1,3299,3299,Medium
3,O0004,Machine Learning Basics,3,999,2997,Medium
4,O0005,Coffee Maker,5,3499,17495,High


Organize the Final Processed DataFrame

In [11]:
final_columns = [
    "Order_ID", "Order_Date", "Order_Year", "Order_Month", "Order_Month_Name",
    "Order_Day", "Order_Day_Name", "Customer_ID", "Customer_Name", "City",
    "Region", "Membership_Type", "Product_ID", "Product_Name", "Category",
    "Brand", "Unit_Price", "Quantity", "Total_Amount", "Order_Value_Category",
    "Payment_Method", "Order_Status"
]

processed = (
    processed[final_columns]
    .sort_values("Order_Date")
    .reset_index(drop=True)
)

processed.head()


,Order_ID,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name,Customer_ID,Customer_Name,City,...,Product_ID,Product_Name,Category,Brand,Unit_Price,Quantity,Total_Amount,Order_Value_Category,Payment_Method,Order_Status
0,O0051,2026-01-01,2026,1,January,1,Thursday,C008,Sara Ahmed,Hyderabad,...,P017,Yoga Mat,Sports,FitLife,899,4,3596,Medium,Net Banking,Delivered
1,O0091,2026-01-01,2026,1,January,1,Thursday,C002,Zoya Khan,Delhi,...,P011,Air Fryer,Home & Kitchen,CookSmart,4999,2,9998,High,Credit Card,Delivered
2,O0022,2026-01-02,2026,1,January,2,Friday,C023,Nikhil Sood,Chandigarh,...,P014,Data Science Handbook,Books,DataPress,899,4,3596,Medium,Net Banking,Delivered
3,O0106,2026-01-02,2026,1,January,2,Friday,C001,Aarav Sharma,Srinagar,...,P009,Coffee Maker,Home & Kitchen,HomeBrew,3499,1,3499,Medium,Net Banking,Delivered
4,O0076,2026-01-03,2026,1,January,3,Saturday,C019,Yusuf Dar,Srinagar,...,P015,Machine Learning Basics,Books,AIPress,999,1,999,Low,Debit Card,Delivered


 Basic Validation and Summary

In [12]:
print("Rows:", len(processed))
print("Columns:", len(processed.columns))
print("Missing values:")
print(processed.isna().sum())

print("\nTotal sales amount:", processed["Total_Amount"].sum())
print("\nOrders by status:")
print(processed["Order_Status"].value_counts())


Rows: 120
Columns: 22
Missing values:
Order_ID                0
Order_Date              0
Order_Year              0
Order_Month             0
Order_Month_Name        0
Order_Day               0
Order_Day_Name          0
Customer_ID             0
Customer_Name           0
City                    0
Region                  0
Membership_Type         0
Product_ID              0
Product_Name            0
Category                0
Brand                   0
Unit_Price              0
Quantity                0
Total_Amount            0
Order_Value_Category    0
Payment_Method          0
Order_Status            0
dtype: int64

Total sales amount: 741507

Orders by status:
Order_Status
Delivered    79
Cancelled    25
Shipped      16
Name: count, dtype: int64


 Export the Final Processed Dataset

In [13]:
output_file = "Day9_Processed_Ecommerce.csv"
processed.to_csv(output_file, index=False)

print(f"Processed dataset exported successfully to: {output_file}")


Processed dataset exported successfully to: Day9_Processed_Ecommerce.csv


 Final Dataset Preview

In [14]:
processed.head(10)


,Order_ID,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name,Customer_ID,Customer_Name,City,...,Product_ID,Product_Name,Category,Brand,Unit_Price,Quantity,Total_Amount,Order_Value_Category,Payment_Method,Order_Status
0,O0051,2026-01-01,2026,1,January,1,Thursday,C008,Sara Ahmed,Hyderabad,...,P017,Yoga Mat,Sports,FitLife,899,4,3596,Medium,Net Banking,Delivered
1,O0091,2026-01-01,2026,1,January,1,Thursday,C002,Zoya Khan,Delhi,...,P011,Air Fryer,Home & Kitchen,CookSmart,4999,2,9998,High,Credit Card,Delivered
2,O0022,2026-01-02,2026,1,January,2,Friday,C023,Nikhil Sood,Chandigarh,...,P014,Data Science Handbook,Books,DataPress,899,4,3596,Medium,Net Banking,Delivered
3,O0106,2026-01-02,2026,1,January,2,Friday,C001,Aarav Sharma,Srinagar,...,P009,Coffee Maker,Home & Kitchen,HomeBrew,3499,1,3499,Medium,Net Banking,Delivered
4,O0076,2026-01-03,2026,1,January,3,Saturday,C019,Yusuf Dar,Srinagar,...,P015,Machine Learning Basics,Books,AIPress,999,1,999,Low,Debit Card,Delivered
5,O0108,2026-01-04,2026,1,January,4,Sunday,C006,Ishita Gupta,Bengaluru,...,P020,Dumbbell Set,Sports,StrongFit,1999,1,1999,Low,Net Banking,Delivered
6,O0041,2026-01-04,2026,1,January,4,Sunday,C008,Sara Ahmed,Hyderabad,...,P012,Water Bottle,Home & Kitchen,HydroLife,699,5,3495,Medium,Cash on Delivery,Delivered
7,O0018,2026-01-05,2026,1,January,5,Monday,C003,Rohan Mehta,Mumbai,...,P011,Air Fryer,Home & Kitchen,CookSmart,4999,4,19996,High,Net Banking,Cancelled
8,O0104,2026-01-05,2026,1,January,5,Monday,C011,Vivaan Kapoor,Jaipur,...,P006,Hoodie,Clothing,UrbanWear,1599,3,4797,Medium,Debit Card,Shipped
9,O0110,2026-01-05,2026,1,January,5,Monday,C007,Aditya Verma,Pune,...,P005,Power Bank,Electronics,VoltPlus,1199,1,1199,Low,UPI,Delivered
